# Step 1. Introduction & Final Analysis Objective

The previous notebooks developed two forecasting approaches for Rossmann store sales: a Machine Learning approach using Random Forest and a Deep Learning approach using LSTM.

This notebook brings both approaches together to compare their performance, select the better-performing model, and generate the final test-set sales predictions.

# Step 2. Import Libraries & Load Data

In [ ]:
import os
import logging
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import load_model

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Libraries imported successfully.")

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")
store_df = pd.read_csv("../data/store.csv")
sample_submission = pd.read_csv("../data/sample_submission.csv")

logging.info("All datasets loaded successfully.")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Store shape:", store_df.shape)
print("Sample submission shape:", sample_submission.shape)

Train shape: (1017209, 9)
Test shape: (41088, 8)
Store shape: (1115, 10)
Sample submission shape: (41088, 2)


# Step 3. Load Trained Models
The trained Random Forest pipeline and LSTM model saved in the previous notebooks will be loaded for final analysis. The LSTM scaler is also loaded because it is required to transform LSTM inputs and convert its predictions back to the original sales scale.

In [ ]:
project_folder = ".."

# Model paths
rf_model_path = os.path.join(
    project_folder,
    "models",
    "final_random_forest_pipeline.pkl"
)

lstm_model_path = os.path.join(
    project_folder,
    "models",
    "final_lstm_model.keras"
)

lstm_scaler_path = os.path.join(
    project_folder,
    "models",
    "lstm_scaler.pkl"
)

# Load trained models and scaler
rf_pipeline = joblib.load(rf_model_path)
lstm_model = load_model(lstm_model_path)
lstm_scaler = joblib.load(lstm_scaler_path)

print("Random Forest pipeline loaded successfully.")
print("LSTM model loaded successfully.")
print("LSTM scaler loaded successfully.")

logging.info("All trained models and scaler loaded successfully.")

# Step 4. Prepare Validation Data

Before comparing the two models, we need to make their validation predictions comparable.

The Random Forest model predicts sales at the store-date level, while the LSTM model predicts daily total sales. Therefore, Random Forest validation predictions will be aggregated by date so that both models are evaluated on the same daily total-sales level.

We will recreate the Random Forest validation data using the same feature preparation approach used in Notebook 2.

In [3]:
# Copy and merge training data
train_model = train_df.copy()

train_model = train_model.merge(
    store_df,
    on="Store",
    how="left"
)

# Date features
train_model["Date"] = pd.to_datetime(train_model["Date"])
train_model["Year"] = train_model["Date"].dt.year
train_model["Month"] = train_model["Date"].dt.month
train_model["Day"] = train_model["Date"].dt.day
train_model["WeekOfYear"] = (
    train_model["Date"].dt.isocalendar().week.astype(int)
)
train_model["IsWeekend"] = (
    train_model["DayOfWeek"] >= 6
).astype(int)

# Promo2-related missing values
train_model["Promo2SinceWeek"] = (
    train_model["Promo2SinceWeek"].fillna(0)
)

train_model["Promo2SinceYear"] = (
    train_model["Promo2SinceYear"].fillna(0)
)

train_model["PromoInterval"] = (
    train_model["PromoInterval"].fillna("None")
)

# Competition distance
competition_distance_median = (
    train_model["CompetitionDistance"].median()
)

train_model["CompetitionDistance"] = (
    train_model["CompetitionDistance"]
    .fillna(competition_distance_median)
)

# Competition opening information
train_model["CompetitionOpenSinceMonth"] = (
    train_model["CompetitionOpenSinceMonth"].fillna(0)
)

train_model["CompetitionOpenSinceYear"] = (
    train_model["CompetitionOpenSinceYear"].fillna(0)
)

# Competition duration
train_model["CompetitionOpenMonths"] = 0

competition_available = (
    (train_model["CompetitionOpenSinceYear"] > 0) &
    (train_model["CompetitionOpenSinceMonth"] > 0)
)

train_model.loc[
    competition_available,
    "CompetitionOpenMonths"
] = (
    (
        train_model.loc[competition_available, "Year"]
        - train_model.loc[competition_available, "CompetitionOpenSinceYear"]
    ) * 12
    +
    (
        train_model.loc[competition_available, "Month"]
        - train_model.loc[competition_available, "CompetitionOpenSinceMonth"]
    )
).clip(lower=0)

# Promo2 participation
train_model["Promo2Active"] = (
    train_model["Promo2"].astype(int)
)

# StateHoliday standardization
train_model["StateHoliday"] = (
    train_model["StateHoliday"]
    .astype(str)
    .replace({
        "0": "None",
        "0.0": "None"
    })
)

# Remove unavailable/target features
X_rf = train_model.drop(
    columns=["Sales", "Customers"]
)

y_rf = train_model["Sales"]

# Chronological split
sort_index = train_model["Date"].sort_values().index

X_rf = X_rf.loc[sort_index]
y_rf = y_rf.loc[sort_index]

split_index_rf = int(len(X_rf) * 0.80)

X_rf_val = X_rf.iloc[split_index_rf:].copy()
y_rf_val = y_rf.iloc[split_index_rf:].copy()

print("RF validation samples:", len(X_rf_val))
print(
    "RF validation period:",
    X_rf_val["Date"].min(),
    "to",
    X_rf_val["Date"].max()
)

RF validation samples: 203442
RF validation period: 2015-01-30 00:00:00 to 2015-07-31 00:00:00


# Step 5. Compare ML vs LSTM

## Random Forest Daily Validation Predictions

Random Forest generates predictions for each store and date. These predictions are then summed by date so that they represent total daily sales, matching the level used by the LSTM.

In [4]:
# Generate RF validation predictions
rf_val_pred = rf_pipeline.predict(X_rf_val)

# Store-level validation results
rf_validation_results = X_rf_val[["Date"]].copy()

rf_validation_results["Actual_Sales"] = y_rf_val.values
rf_validation_results["Predicted_Sales"] = rf_val_pred

# Aggregate store-level predictions to daily total sales
rf_daily_results = (
    rf_validation_results
    .groupby("Date", as_index=False)
    .agg(
        Actual_Sales=("Actual_Sales", "sum"),
        RF_Predicted_Sales=("Predicted_Sales", "sum")
    )
)

print("RF daily validation shape:", rf_daily_results.shape)

display(rf_daily_results.head())

RF daily validation shape: (183, 3)


,Date,Actual_Sales,RF_Predicted_Sales
0,2015-01-30,4344696,3.711535e+06
1,2015-01-31,7354597,6.482214e+06
2,2015-02-01,225556,1.903755e+05
3,2015-02-02,10291172,1.042001e+07
4,2015-02-03,8904680,8.862923e+06


## Generate LSTM Validation Predictions

The LSTM was trained using the aggregated daily sales series. We recreate the same 7-day sequences and chronological validation period used in Notebook 3.

In [5]:
# Create daily total sales
daily_sales = (
    train_df.groupby("Date", as_index=False)["Sales"]
    .sum()
    .sort_values("Date")
    .set_index("Date")
)

sales_series = daily_sales["Sales"]

# Same look-back used in Notebook 3
look_back = 7

# Create sequences
def create_sequences(data, look_back):
    X, y = [], []

    for i in range(look_back, len(data)):
        X.append(data[i - look_back:i])
        y.append(data[i])

    return np.array(X), np.array(y)


sales_values = sales_series.values

X_lstm, y_lstm = create_sequences(
    sales_values,
    look_back
)

# Chronological split
split_index_lstm = int(len(X_lstm) * 0.80)

X_lstm_train = X_lstm[:split_index_lstm]
X_lstm_val = X_lstm[split_index_lstm:]

y_lstm_train = y_lstm[:split_index_lstm]
y_lstm_val = y_lstm[split_index_lstm:]

# Scale using the saved scaler
X_lstm_val_scaled = lstm_scaler.transform(
    X_lstm_val.reshape(-1, 1)
).reshape(X_lstm_val.shape[0], X_lstm_val.shape[1], 1)

# Generate predictions
lstm_pred_scaled = lstm_model.predict(
    X_lstm_val_scaled,
    verbose=0
)

# Convert predictions back to original sales scale
lstm_pred = lstm_scaler.inverse_transform(
    lstm_pred_scaled
).flatten()

# Validation dates corresponding to LSTM targets
lstm_validation_dates = sales_series.index[
    look_back + split_index_lstm:
]

lstm_daily_results = pd.DataFrame({
    "Date": lstm_validation_dates,
    "Actual_Sales": y_lstm_val,
    "LSTM_Predicted_Sales": lstm_pred
})

print("LSTM daily validation shape:", lstm_daily_results.shape)

display(lstm_daily_results.head())

LSTM daily validation shape: (187, 3)


,Date,Actual_Sales,LSTM_Predicted_Sales
0,2015-01-26,9401803,7710147.5
1,2015-01-27,8392342,7748430.5
2,2015-01-28,7930037,7806103.5
3,2015-01-29,8024066,7812194.0
4,2015-01-30,9454954,7628919.0


## Align RF and LSTM Validation Periods
Both models must be evaluated on the same dates. We therefore merge their daily validation results using the common validation dates.

In [6]:
# Make sure Date has the same datetime format in both results
rf_daily_results["Date"] = pd.to_datetime(
    rf_daily_results["Date"]
)

lstm_daily_results["Date"] = pd.to_datetime(
    lstm_daily_results["Date"]
)

# Merge RF and LSTM results on Date
comparison_df = pd.merge(
    rf_daily_results,
    lstm_daily_results,
    on="Date",
    how="inner",
    suffixes=("_RF", "_LSTM")
)

# Keep one actual-sales column
comparison_df = comparison_df[
    [
        "Date",
        "Actual_Sales_RF",
        "RF_Predicted_Sales",
        "LSTM_Predicted_Sales"
    ]
].rename(
    columns={
        "Actual_Sales_RF": "Actual_Sales"
    }
)

print("Common validation dates:", len(comparison_df))

print(
    "Comparison period:",
    comparison_df["Date"].min(),
    "to",
    comparison_df["Date"].max()
)

display(comparison_df.head())

Common validation dates: 183
Comparison period: 2015-01-30 00:00:00 to 2015-07-31 00:00:00


,Date,Actual_Sales,RF_Predicted_Sales,LSTM_Predicted_Sales
0,2015-01-30,4344696,3.711535e+06,7.628919e+06
1,2015-01-31,7354597,6.482214e+06,5.468006e+06
2,2015-02-01,225556,1.903755e+05,-2.954439e+05
3,2015-02-02,10291172,1.042001e+07,6.912174e+06
4,2015-02-03,8904680,8.862923e+06,7.039139e+06


## Compare MAE and RMSE
Both models are now evaluated against the same daily total sales values and the same validation dates. MAE and RMSE are calculated automatically so that the better-performing model is selected based on actual validation performance.

In [7]:
# Calculate Random Forest metrics
rf_daily_mae = mean_absolute_error(
    comparison_df["Actual_Sales"],
    comparison_df["RF_Predicted_Sales"]
)

rf_daily_rmse = np.sqrt(
    mean_squared_error(
        comparison_df["Actual_Sales"],
        comparison_df["RF_Predicted_Sales"]
    )
)

# Calculate LSTM metrics
lstm_daily_mae = mean_absolute_error(
    comparison_df["Actual_Sales"],
    comparison_df["LSTM_Predicted_Sales"]
)

lstm_daily_rmse = np.sqrt(
    mean_squared_error(
        comparison_df["Actual_Sales"],
        comparison_df["LSTM_Predicted_Sales"]
    )
)

comparison_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "LSTM"
    ],
    "MAE": [
        rf_daily_mae,
        lstm_daily_mae
    ],
    "RMSE": [
        rf_daily_rmse,
        lstm_daily_rmse
    ]
})

display(comparison_results)

,Model,MAE,RMSE
0,Random Forest,3.670347e+05,6.177317e+05
1,LSTM,1.431762e+06,2.278280e+06


Observation

* Random Forest clearly performs better than LSTM on the common validation period.
* RF MAE is about ₹3.67 lakh, while LSTM MAE is about ₹14.32 lakh.
* RF RMSE is about ₹6.18 lakh, compared with LSTM's ₹22.78 lakh.
* Lower MAE and RMSE are better, so both metrics select Random Forest.
* The LSTM also produces unrealistic negative predictions, such as -₹2.95 lakh on 2015-02-01.
* The validation dates are properly aligned: 2015-01-30 to 2015-07-31, with 183 common dates.
* The RF predictions were aggregated from store-level predictions to daily totals, making them comparable with the LSTM's daily aggregate predictions.

## Select Best Model

In [8]:
best_model = comparison_results.loc[
    comparison_results["RMSE"].idxmin(), "Model"
]

print("Best validation model:", best_model)

Best validation model: Random Forest


Observation

Random Forest is selected as the best-performing model because it achieved the lowest MAE and RMSE on the common validation period. It provides substantially more accurate sales forecasts than the LSTM model.

# Step 6. Test Data Prediction

Now that the Random Forest model has been selected as the final ML model, we can move from validation to the actual forecasting stage.

In this step, we will prepare the test dataset using the same feature engineering and preprocessing logic used during model training. This ensures that the test data has the same feature structure expected by the trained Random Forest pipeline.

Once the test data is prepared, the trained Random Forest model will generate sales predictions for every Store-Date combination in test.csv. These predictions represent the model's forecast of future Rossmann sales.

In [10]:
# Create a copy of test data and merge store information
test_model = test_df.copy()

test_model = test_model.merge(
    store_df,
    on="Store",
    how="left"
)

# Date-based features
test_model["Date"] = pd.to_datetime(test_model["Date"])
test_model["Year"] = test_model["Date"].dt.year
test_model["Month"] = test_model["Date"].dt.month
test_model["Day"] = test_model["Date"].dt.day
test_model["WeekOfYear"] = test_model["Date"].dt.isocalendar().week.astype(int)
test_model["IsWeekend"] = (test_model["DayOfWeek"] >= 6).astype(int)

# Promo2 features
test_model["Promo2SinceWeek"] = test_model["Promo2SinceWeek"].fillna(0)
test_model["Promo2SinceYear"] = test_model["Promo2SinceYear"].fillna(0)
test_model["PromoInterval"] = test_model["PromoInterval"].fillna("None")

# Competition distance
test_model["CompetitionDistance"] = test_model["CompetitionDistance"].fillna(
    competition_distance_median
)

# Competition start date
test_model["CompetitionOpenSinceMonth"] = (
    test_model["CompetitionOpenSinceMonth"].fillna(0)
)

test_model["CompetitionOpenSinceYear"] = (
    test_model["CompetitionOpenSinceYear"].fillna(0)
)

# Competition open duration
test_model["CompetitionOpenMonths"] = 0

competition_available = (
    (test_model["CompetitionOpenSinceYear"] > 0) &
    (test_model["CompetitionOpenSinceMonth"] > 0)
)

test_model.loc[competition_available, "CompetitionOpenMonths"] = (
    (
        test_model.loc[competition_available, "Year"]
        - test_model.loc[competition_available, "CompetitionOpenSinceYear"]
    ) * 12
    +
    (
        test_model.loc[competition_available, "Month"]
        - test_model.loc[competition_available, "CompetitionOpenSinceMonth"]
    )
).clip(lower=0)

# Promo2 active feature
test_model["Promo2Active"] = test_model["Promo2"].astype(int)

# Standardize StateHoliday values
test_model["StateHoliday"] = (
    test_model["StateHoliday"]
    .astype(str)
    .replace({"0": "None", "0.0": "None"})
)

# Handle missing Open values using the same value already prepared
# in the ML workflow
test_model["Open"] = test_model["Open"].fillna(1)

# Prepare model input
X_test = test_model.drop(columns=["Id"])

print("Test model shape:", X_test.shape)
print("Test prediction samples:", len(X_test))
print("Missing values in model input:", X_test.isnull().sum().sum())

Test model shape: (41088, 23)
Test prediction samples: 41088
Missing values in model input: 0


Observation
* Test data contains 41,088 records, matching the original test.csv.
* After merging with store information and feature engineering, there are 23 model input features.
* All 41,088 test records are ready for prediction.
* There are 0 missing values, so the trained Random Forest pipeline can be applied directly.
* The preprocessing is consistent with the ML workflow used during training.

# Step 7. Submission File

## Generate Final Sales Predictions

The Random Forest model has been selected as the final model. We will now use it to predict sales for all records in the test dataset.

In [11]:
# Generate final test predictions
test_predictions = rf_pipeline.predict(X_test)

# Ensure predictions are non-negative
test_predictions = np.clip(test_predictions, 0, None)

print("Prediction shape:", test_predictions.shape)
print("Minimum predicted sales:", test_predictions.min())
print("Maximum predicted sales:", test_predictions.max())
print("Mean predicted sales:", test_predictions.mean())

Prediction shape: (41088,)
Minimum predicted sales: 0.0
Maximum predicted sales: 31349.51
Mean predicted sales: 5760.1892045563945


Observation
* The model generated predictions for all 41,088 test records.
* Prediction shape (41088,) matches the number of rows in test.csv.
* The minimum prediction is 0, so there are no negative sales predictions after clipping.
* Maximum predicted sales is approximately 31,349.51.
* Average predicted sales is approximately 5,760.19.
* The predictions are ready to be placed into the sample_submission.csv format.

## Create Submission DataFrame

Now we will use sample_submission.csv as the submission template and replace its Sales column with our Random Forest predictions. This keeps the required Id structure exactly as provided by the competition.

In [12]:
# Create final submission using the sample submission template
submission = sample_submission.copy()

submission["Sales"] = test_predictions

print("Submission shape:", submission.shape)
display(submission.head())

Submission shape: (41088, 2)


,Id,Sales
0,1,4383.58
1,2,9813.92
2,3,9395.29
3,4,7159.17
4,5,6860.44


Observation
* Submission shape is (41,088, 2).
* The submission contains the required Id and Sales columns.
* Sales predictions have been successfully inserted into the submission template.
* The Id values are preserved from sample_submission.csv.
* The file is now ready for final validation before saving.

## Validate Id
Before saving the final submission, we should verify that the Id column has not been changed. The submission must maintain exactly the same IDs and order as the provided sample_submission.csv.

In [13]:
id_match = submission["Id"].equals(sample_submission["Id"])

print("Id values and order match:", id_match)
print("Number of unique Ids:", submission["Id"].nunique())
print("Expected Id count:", len(sample_submission))

Id values and order match: True
Number of unique Ids: 41088
Expected Id count: 41088


Observation
* Id values and their order exactly match the provided sample_submission.csv.
* All 41,088 IDs are unique.
* The number of unique IDs matches the expected count of 41,088.
* Therefore, the submission structure is correctly preserved.

## Validate Sales Predictions

Next, we will check the Sales column to make sure there are no missing, infinite, or negative predictions before saving the final submission.

In [14]:
print("Missing Sales:", submission["Sales"].isnull().sum())
print("Infinite Sales:", np.isinf(submission["Sales"]).sum())
print("Negative Sales:", (submission["Sales"] < 0).sum())
print("Total Sales Predictions:", len(submission["Sales"]))

Missing Sales: 0
Infinite Sales: 0
Negative Sales: 0
Total Sales Predictions: 41088


Observation
* 0 missing sales predictions.
* 0 infinite values.
* 0 negative predictions.
* All 41,088 test records have valid sales predictions.
* The submission is now fully validated and ready to save.

## Save Final Submission

All required validations have passed. We will now save the final prediction file as a CSV so it can be submitted to the Rossmann Store Sales competition.

In [15]:
submission_path = "rossmann_final_submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("Final submission saved successfully.")
print("File:", submission_path)
print("Submission shape:", submission.shape)

Final submission saved successfully.
File: rossmann_final_submission.csv
Submission shape: (41088, 2)


# Step 8. Final Business Insights

The complete analysis across EDA, machine learning, and deep learning provides several business insights for Rossmann store sales forecasting.

* Promotions influence sales: Promotional activities are an important factor affecting store sales and should be considered when planning inventory and staffing.
* Customer count is strongly related to sales: Stores with higher customer traffic generally generate higher sales, making customer behavior an important indicator of store performance.
* Sales show strong time-based patterns: Day of week, month, and seasonal patterns influence daily sales.
* Store characteristics matter: Store type, assortment, and competition-related features contribute to differences in sales between stores.
* Competition can affect store performance: Competition distance and the duration for which a competitor has been present can influence store sales.
* Historical sales patterns are useful for forecasting: The analysis showed clear temporal patterns that can be used to predict future sales.
* Random Forest performed better than the LSTM model on the common validation period, achieving substantially lower MAE and RMSE.
* Random Forest was selected for the final submission because it provides direct Store-Date level predictions required by the competition dataset.
* The final model generated predictions for 41,088 test records.

# Step 9. Final Model & Prediction Logging

In [16]:
logging.info("Final model comparison completed.")
logging.info(f"Random Forest validation MAE: {rf_daily_mae:.2f}")
logging.info(f"Random Forest validation RMSE: {rf_daily_rmse:.2f}")
logging.info(f"LSTM validation MAE: {lstm_daily_mae:.2f}")
logging.info(f"LSTM validation RMSE: {lstm_daily_rmse:.2f}")
logging.info("Random Forest selected as the final submission model.")
logging.info(f"Test predictions generated: {len(test_predictions)}")
logging.info(f"Final submission saved: {submission_path}")

# Step 10. Conclusion

* The Rossmann dataset was analyzed using exploratory data analysis and feature engineering.
* A Random Forest regression pipeline was developed using store, promotion, competition, holiday, and date-related features.
* An LSTM model was developed to learn sequential daily sales patterns.
Both models were evaluated on a common validation period from 30 January 2015 to 31 July 2015.
* Random Forest achieved:
  * MAE: approximately 367,035
  * RMSE: approximately 617,732
* LSTM achieved:
  * MAE: approximately 1,431,762
  * RMSE: approximately 2,278,280
* Therefore, Random Forest was selected as the final submission model.
* The final model generated predictions for all 41,088 test records.
* The submission was validated for: Correct IDs,Correct number of records,Missing values,Infinite values,Negative predictions
* The final submission file rossmann_final_submission.csv was successfully created.
* The trained models and preprocessing artifacts can be reused for future predictions.